In [22]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path
import os

print("transformers version:", __import__('transformers').__version__)
print("torch version:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
transformers version: 5.0.0
torch version: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB


In [23]:
MODEL_ID = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="eager"
)

model = model.to("cuda")
model.eval()

n_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
vocab_size = model.config.vocab_size

print(f"\nmodel successfully loaded")
print(f"Layers: {n_layers}")
print(f"Hidden size: {hidden_size}")
print(f"Vocab size: {vocab_size}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


model successfully loaded
Layers: 32
Hidden size: 4096
Vocab size: 32000
Parameters: 7,241,732,096


In [24]:
test_input = "The most common cause of myocardial infarction is"
inputs = tokenizer(test_input, return_tensors="pt").to("cuda")

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print("Input:", test_input)
print("Output:", tokenizer.decode(out[0], skip_special_tokens=True))
print("\nBasic generation confirmed to be working")

Input: The most common cause of myocardial infarction is
Output: The most common cause of myocardial infarction is atherosclerosis, which is a

Basic generation confirmed to be working


In [25]:
hook_storage = {
    "hidden_states": {},
    "attention_maps": {},
    "logits_per_layer": {}
}
hooks = []

def make_hidden_state_hook(layer_idx):
    def hook(module, input, output):
        hidden = output[0].detach().squeeze(0)
        hook_storage["hidden_states"][layer_idx] = hidden
        with torch.no_grad():
            normed = model.model.norm(hidden)
            logits = model.lm_head(normed)
        hook_storage["logits_per_layer"][layer_idx] = logits.detach()
    return hook

for i, block in enumerate(model.model.layers):
    h = block.register_forward_hook(make_hidden_state_hook(i))
    hooks.append(h)

print(f"Registered hooks on {n_layers} transformer blocks.")
print("Attention maps will be read directly from outputs.attentions")

Registered hooks on 32 transformer blocks.
Attention maps will be read directly from outputs.attentions


In [26]:
query = "What is the mechanism of action of metformin in type 2 diabetes?"
inputs = tokenizer(
    query,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to("cuda")
seq_len = inputs["input_ids"].shape[1]

for key in hook_storage:
    hook_storage[key].clear()

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

for i, attn in enumerate(outputs.attentions):
    hook_storage["attention_maps"][i] = attn.detach().squeeze(0)

print(f"hidden_states:    {len(hook_storage['hidden_states'])} layers")
print(f"attention_maps:   {len(hook_storage['attention_maps'])} layers")
print(f"logits_per_layer: {len(hook_storage['logits_per_layer'])} layers")

hidden_states:    32 layers
attention_maps:   32 layers
logits_per_layer: 32 layers


In [27]:
print("=== TENSOR SHAPE VERIFICATION ===\n")

all_good = True

for layer_idx in range(n_layers):
    hidden = hook_storage["hidden_states"][layer_idx]
    attn = hook_storage["attention_maps"][layer_idx]
    logits = hook_storage["logits_per_layer"][layer_idx]

    expected_hidden = (seq_len, hidden_size)
    expected_attn = (model.config.num_attention_heads, seq_len, seq_len)
    expected_logits = (seq_len, vocab_size)

    if tuple(hidden.shape) != expected_hidden:
        print(f"Layer {layer_idx}: hidden shape mismatch {tuple(hidden.shape)} != {expected_hidden}")
        all_good = False

    if tuple(attn.shape) != expected_attn:
        print(f"Layer {layer_idx}: attention shape mismatch {tuple(attn.shape)} != {expected_attn}")
        all_good = False

    if tuple(logits.shape) != expected_logits:
        print(f"Layer {layer_idx}: logits shape mismatch {tuple(logits.shape)} != {expected_logits}")
        all_good = False

print(f"\nAll tensor shapes correct: {all_good}")

=== TENSOR SHAPE VERIFICATION ===


All tensor shapes correct: True


In [28]:
DATA_PATH = "/content/drive/MyDrive/medrag/pubmedqa_filtered.json"

with open(DATA_PATH, "r") as f:
    pubmed_data = json.load(f)

sample = pubmed_data[0]

question = sample["query"]
abstract = sample["supporting_abstracts"][0]

prompt = f"Context:\n{abstract}\n\nQuestion: {question}\nAnswer:"
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to("cuda")

for key in hook_storage:
    hook_storage[key].clear()

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

for i, attn in enumerate(outputs.attentions):
    hook_storage["attention_maps"][i] = attn.detach().squeeze(0)

print("Sample PubMedQA prompt confirmed")
print("Question:", question)
print("Input tokens:", inputs["input_ids"].shape[1])
print("Hidden states captured:", len(hook_storage["hidden_states"]))
print("Attention maps captured:", len(hook_storage["attention_maps"]))
print("Logits captured:", len(hook_storage["logits_per_layer"]))

Sample PubMedQA prompt confirmed
Question: Landolt C and snellen e acuity: differences in strabismus amblyopia?
Input tokens: 123
Hidden states captured: 32
Attention maps captured: 32
Logits captured: 32


In [29]:
OUTPUT_DIR = "/content/drive/MyDrive/medrag/biomistral_model_load_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

report = {
    "model_id": MODEL_ID,
    "model_class": model.__class__.__name__,
    "n_layers": n_layers,
    "hidden_size": hidden_size,
    "vocab_size": vocab_size,
    "num_attention_heads": model.config.num_attention_heads,
    "max_length": 512,
    "dtype": "float16",
    "attn_implementation": "eager",
    "generation_confirmed": True,
    "hooks_confirmed": len(hook_storage["hidden_states"]) == n_layers,
    "attention_confirmed": len(hook_storage["attention_maps"]) == n_layers,
    "logits_confirmed": len(hook_storage["logits_per_layer"]) == n_layers,
    "tensor_shapes_confirmed": all_good
}

report_path = os.path.join(OUTPUT_DIR, "model_load_report.json")

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

for h in hooks:
    h.remove()

print(f"Report saved to {report_path}")
print("00_biomistral_model_load_test complete")

Report saved to /content/drive/MyDrive/medrag/biomistral_model_load_test/model_load_report.json
00_biomistral_model_load_test complete
